In [1]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)

from dotenv import load_dotenv
load_dotenv()

from datetime import datetime, timedelta
from quantrl_lab.data import DataProcessor
from quantrl_lab.data.source_registry import DataSourceRegistry
from quantrl_lab.data.processing.sentiment import HuggingFaceConfig, HuggingFaceProvider, SentimentConfig

In [2]:
symbol = "AAPL"
end_date = datetime.now()
start_date = end_date - timedelta(days=1000)


registry = DataSourceRegistry()
# alpaca is the default for both olhcv + news so we do not have to define another loader for news
alpaca_loader = registry.primary_source
fmp_loader = registry.get_source("fundamental_source")

In [3]:
ohlcv_df = alpaca_loader.get_historical_ohlcv_data(
        symbols=symbol,
        start=start_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        timeframe="1d", # there is support for intraday as well
    )

2026-02-07 15:56:05.191 | INFO     | quantrl_lab.data.sources.alpaca_loader:get_historical_ohlcv_data:157 - Fetching historical data for ['AAPL'] from 2023-05-14 00:00:00 to 2026-02-07 00:00:00 with timeframe 1d
2026-02-07 15:56:06.164 | SUCCESS  | quantrl_lab.data.utils.response_validation:log_dataframe_info:190 - Fetched OHLCV data for 1 symbol(s): 686 rows


In [4]:
news_df = alpaca_loader.get_news_data(
        symbols=symbol,
        start=start_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        verbose = False
    )

In [5]:
profile = fmp_loader.get_company_profile(symbol)
sector = profile.iloc[0].get('sector') if not profile.empty else None
industry = profile.iloc[0].get('industry') if not profile.empty else None
print(f"  Sector: {sector}, Industry: {industry}")

2026-02-07 15:59:13.449 | INFO     | quantrl_lab.data.sources.fmp_loader:get_company_profile:550 - Fetching company profile for: AAPL
2026-02-07 15:59:13.449 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:161 - HTTP GET request to https://financialmodelingprep.com/stable/profile (attempt 1/4)
2026-02-07 15:59:14.437 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:191 - Request successful: 2565 bytes
2026-02-07 15:59:14.438 | DEBUG    | quantrl_lab.data.utils.response_validation:convert_to_dataframe_safe:119 - Successfully converted response to DataFrame with 1 rows for symbol: AAPL
2026-02-07 15:59:14.438 | SUCCESS  | quantrl_lab.data.sources.fmp_loader:get_company_profile:576 - Fetched company profile for AAPL: Apple Inc. (Technology - Consumer Electronics)


  Sector: Technology, Industry: Consumer Electronics


In [6]:
grades_df = fmp_loader.get_historical_grades(symbol) # It is at the monthly granularity

ratings_df = fmp_loader.get_historical_rating(symbol, limit=1000) # does not have from and to to control dates

2026-02-07 15:59:14.444 | DEBUG    | quantrl_lab.data.utils.request_utils:_apply_rate_limit:68 - Rate limiting: sleeping for 0.99s
2026-02-07 15:59:15.438 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:161 - HTTP GET request to https://financialmodelingprep.com/stable/grades-historical (attempt 1/4)
2026-02-07 15:59:16.535 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:191 - Request successful: 15419 bytes
2026-02-07 15:59:16.538 | SUCCESS  | quantrl_lab.data.sources.fmp_loader:get_historical_grades:299 - Fetched 86 historical grades for AAPL
2026-02-07 15:59:16.538 | DEBUG    | quantrl_lab.data.utils.request_utils:_apply_rate_limit:68 - Rate limiting: sleeping for 1.00s
2026-02-07 15:59:17.535 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:161 - HTTP GET request to https://financialmodelingprep.com/stable/ratings-historical (attempt 1/4)
2026-02-07 15:59:19.177 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:191 - Request 

In [7]:
ratings_df.head()

,symbol,date,rating,overallScore,discountedCashFlowScore,returnOnEquityScore,returnOnAssetsScore,debtToEquityScore,priceToEarningsScore,priceToBookScore
999,AAPL,2022-01-10,S,5,5,5,3,5,5,5
998,AAPL,2022-01-11,S,5,5,5,3,5,5,5
997,AAPL,2022-01-14,S,5,5,5,3,5,5,5
996,AAPL,2022-01-21,S,5,5,5,3,5,5,5
995,AAPL,2022-01-24,S,5,5,5,3,5,5,5


In [8]:
grades_df.head()

,symbol,date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell
85,AAPL,2018-12-01,17,14,18,0,0
84,AAPL,2019-01-01,17,14,18,0,0
83,AAPL,2019-02-01,15,10,22,1,0
82,AAPL,2019-03-01,15,10,22,1,0
81,AAPL,2019-04-01,15,10,22,1,0


In [9]:
sector_perf_df = fmp_loader.get_historical_sector_performance(
    sector, start=start_date.strftime("%Y-%m-%d"), end=end_date.strftime("%Y-%m-%d")
)

industry_perf_df = fmp_loader.get_historical_industry_performance(
    industry, start=start_date.strftime("%Y-%m-%d"), end=end_date.strftime("%Y-%m-%d")
)

2026-02-07 15:59:19.210 | INFO     | quantrl_lab.data.sources.fmp_loader:get_historical_sector_performance:377 - Fetching historical performance for sector: Technology
2026-02-07 15:59:19.210 | DEBUG    | quantrl_lab.data.utils.request_utils:_apply_rate_limit:68 - Rate limiting: sleeping for 0.96s
2026-02-07 15:59:20.174 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:161 - HTTP GET request to https://financialmodelingprep.com/stable/historical-sector-performance (attempt 1/4)
2026-02-07 15:59:21.723 | DEBUG    | quantrl_lab.data.utils.request_utils:make_request:191 - Request successful: 75812 bytes
2026-02-07 15:59:21.724 | DEBUG    | quantrl_lab.data.utils.response_validation:convert_to_dataframe_safe:119 - Successfully converted response to DataFrame with 705 rows for symbol: Technology
2026-02-07 15:59:21.726 | SUCCESS  | quantrl_lab.data.utils.response_validation:log_dataframe_info:190 - Fetched sector performance for Technology: 705 rows
2026-02-07 15:59:21.726 | S

In [10]:
sector_perf_df.head()

,date,sector,exchange,averageChange
704,2023-05-15,Technology,NASDAQ,0.948548
703,2023-05-16,Technology,NASDAQ,0.038925
702,2023-05-17,Technology,NASDAQ,1.299648
701,2023-05-18,Technology,NASDAQ,2.014515
700,2023-05-19,Technology,NASDAQ,-0.215749


In [11]:
industry_perf_df.head()

,date,industry,exchange,averageChange
704,2023-05-15,Consumer Electronics,NASDAQ,-0.290216
703,2023-05-16,Consumer Electronics,NASDAQ,-0.005074
702,2023-05-17,Consumer Electronics,NASDAQ,0.360412
701,2023-05-18,Consumer Electronics,NASDAQ,1.347406
700,2023-05-19,Consumer Electronics,NASDAQ,0.063226


In [12]:
indicators = [
        # Trend indicators
        {"SMA": {"window": [10, 20, 50]}},
        {"EMA": {"window": [12, 26]}},
        # Momentum indicators
        {"RSI": {"window": [14, 21]}},
        "MACD",
        "STOCH",
        # Volatility indicators
        {"BB": {"window": [10, 20], "num_std": [1.5, 2.0]}},
        "ATR",
        # Volume indicators
        "OBV",
    ]

In [13]:
sentiment_config = SentimentConfig(
        text_column="headline",
        date_column="created_at",
        sentiment_score_column="sentiment_score",
    )

hf_config = HuggingFaceConfig(
    model_name="ProsusAI/finbert",
    device=-1,
    max_length=512,
    truncation=True,
    batch_size=32,
)

sentiment_provider = HuggingFaceProvider(hf_config)

In [14]:
total_days = (end_date - start_date).days
train_end = start_date + timedelta(days=int(total_days * 0.6))
val_end = start_date + timedelta(days=int(total_days * 0.8))

split_config = {
    "train": (start_date.strftime("%Y-%m-%d"), train_end.strftime("%Y-%m-%d")),
    "val": (train_end.strftime("%Y-%m-%d"), val_end.strftime("%Y-%m-%d")),
    "test": (val_end.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")),
}

In [15]:
processor = DataProcessor(
        ohlcv_data=ohlcv_df,
        news_data=news_df,
        analyst_grades=grades_df,
        analyst_ratings=ratings_df,
        sector_performance=sector_perf_df,
        industry_performance=industry_perf_df,
        sentiment_config=sentiment_config,
        sentiment_provider=sentiment_provider,
    )

In [16]:
split_data, metadata = processor.data_processing_pipeline(
        indicators=indicators,
        fillna_strategy="exponential_decay", 
        split_config=split_config,
        verbose=False,
        decay_rate=0.8  # Configurable
    )

2026-02-07 15:59:24.681 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 1/6: Technical Indicators
2026-02-07 15:59:24.702 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 2/6: Analyst Estimates Enrichment
2026-02-07 15:59:24.710 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 3/6: Market Context Enrichment
2026-02-07 15:59:24.715 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 4/6: Sentiment Enrichment


Calculating sentiment scores using HuggingFace model...

Device set to use cpu


✓ Sentiment analysis pipeline initialized with model: ProsusAI/finbert

2026-02-07 16:03:38.498 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 5/6: Numeric Conversion
2026-02-07 16:03:38.504 | DEBUG    | quantrl_lab.data.processing.pipeline:execute:90 - Executing step 6/6: Column Cleanup


In [17]:
split_data["train"].head()

,Open,High,Low,Close,Volume,Trade_count,VWAP,SMA_10,SMA_20,SMA_50,EMA_12,EMA_26,RSI_14,RSI_21,MACD_line_12_26,MACD_signal_9,STOCH_%K_14_1,STOCH_%D_3,BB_middle_10,BB_upper_10_1.5,BB_lower_10_1.5,BB_bandwidth_10,BB_upper_10_2.0,BB_lower_10_2.0,BB_middle_20,BB_upper_20_1.5,BB_lower_20_1.5,BB_bandwidth_20,BB_upper_20_2.0,BB_lower_20_2.0,ATR_14,OBV,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,overallScore,discountedCashFlowScore,returnOnEquityScore,returnOnAssetsScore,debtToEquityScore,priceToEarningsScore,priceToBookScore,sector_averageChange,industry_averageChange,sentiment_score
0,193.670,195.640,193.32,194.500,47506279.0,502596.0,194.378548,192.9990,191.77700,184.4058,192.518525,189.965137,64.874750,65.785615,2.553388,2.759599,67.927773,60.389797,192.9990,195.298353,190.699647,0.031770,196.064804,189.933196,191.77700,194.886617,188.667383,0.043239,195.923157,187.630843,2.850224,506029433.0,10,24,7,1,0,5,5,5,3,5,5,5,-1.088839,0.454658,0.757603
1,196.020,197.200,192.55,193.220,47465977.0,570678.0,194.950386,193.2670,191.97550,184.8288,192.626444,190.206238,60.183187,62.666718,2.420206,2.691720,56.921754,61.736887,193.2670,195.166474,191.367526,0.026209,195.799632,190.734368,191.97550,194.986609,188.964391,0.041826,195.990312,187.960688,2.978779,458563456.0,10,24,7,1,0,5,5,5,3,5,5,5,-0.368141,-0.664107,0.781574
2,194.670,196.626,194.14,195.830,48291445.0,519154.0,195.809443,193.7810,192.28750,185.3040,193.119299,190.622813,65.639687,66.107042,2.496486,2.652673,79.363715,68.071081,193.7810,195.492585,192.069415,0.023554,196.063113,191.498887,192.28750,195.437379,189.137621,0.043683,196.487339,188.087661,3.009295,506854901.0,10,24,7,1,0,5,5,5,3,5,5,5,1.664937,1.331314,0.771774
3,196.060,196.490,195.26,196.450,45241255.0,530646.0,196.200202,194.0270,192.41150,185.7792,193.631714,191.054456,66.803481,66.868567,2.577258,2.637590,81.762295,72.682588,194.0270,196.159652,191.894348,0.029311,196.870535,191.183465,192.41150,195.817657,189.005343,0.047207,196.953043,187.869957,2.882203,552096156.0,10,24,7,1,0,5,5,5,3,5,5,5,0.256852,0.316425,0.815426
4,196.235,196.730,195.28,195.605,35306213.0,478272.0,195.861871,194.2145,192.56875,186.1903,193.935297,191.391534,63.639767,64.785484,2.543763,2.618825,69.476744,76.867585,194.2145,196.464118,191.964882,0.030888,197.213991,191.215009,192.56875,196.139572,188.997928,0.049448,197.329847,187.807653,2.779902,516789943.0,10,24,7,1,0,5,5,5,3,5,5,5,-0.158918,-0.428132,0.875528


In [18]:
metadata

{'symbol': 'AAPL',
 'date_ranges': {'train': {'start': '2023-07-26', 'end': '2025-01-03'},
  'val': {'start': '2025-01-03', 'end': '2025-07-22'},
  'test': {'start': '2025-07-22', 'end': '2026-02-06'}},
 'fillna_strategy': 'exponential_decay',
 'technical_indicators': [{'SMA': {'window': [10, 20, 50]}},
  {'EMA': {'window': [12, 26]}},
  {'RSI': {'window': [14, 21]}},
  'MACD',
  'STOCH',
  {'BB': {'window': [10, 20], 'num_std': [1.5, 2.0]}},
  'ATR',
  'OBV'],
 'news_sentiment_applied': True,
 'analyst_data_applied': True,
 'market_context_applied': True,
 'columns_dropped': ['Symbol'],
 'original_shape': (686, 10),
 'final_shapes': {'train': (364, 49), 'val': (136, 49), 'test': (139, 49)}}